In [0]:
from pyspark.sql import SparkSession, functions as F 

from pyspark.sql.types import StringType 

  

spark = SparkSession.builder.appName("day4-assignment").getOrCreate() 

  

orders = (spark.read.option("header", True)

          .csv("/Volumes/jaswanth/default/day-3_assignment/datasets/orders.csv")) 

orders.createOrReplaceTempView("orders") 

  

customers = (spark.read.option("header", True) 

             .csv("/Volumes/jaswanth/default/day-3_assignment/datasets/customers.csv")) 

  

employees = spark.read.schema(""" 

    emp_id INT, name STRING, dept STRING, salary LONG, manager_id INT, 

    address STRUCT<city: STRING, state: STRING, pincode: STRING>, 

    skills ARRAY<STRING>, certifications MAP<STRING, INT>, join_date DATE 

""").json("/Volumes/jaswanth/default/day-3_assignment/datasets/employees.json")

In [0]:
employees.show(3)

+------+-------------+----------------+-------+----------+--------------------+--------------------+--------------------+----------+
|emp_id|         name|            dept| salary|manager_id|             address|              skills|      certifications| join_date|
+------+-------------+----------------+-------+----------+--------------------+--------------------+--------------------+----------+
|   101|  Kavya Kumar|Data Engineering|2450000|      NULL|{Kolkata, WB, 512...|   [databricks, sql]|{aws -> 2026, azu...|2021-08-27|
|   102|Karthik Gupta|              QA|1600000|      NULL|  {Pune, MH, 541689}|[aws, airflow, py...|                  {}|2020-03-22|
|   103|  Ananya Iyer|              QA|2050000|      NULL|{Chennai, TN, 541...|[sql, azure, data...|{databricks -> 2024}|2021-12-21|
+------+-------------+----------------+-------+----------+--------------------+--------------------+--------------------+----------+
only showing top 3 rows


In [0]:
# Task 1.  Same question two ways 

#Dataframe API
result=orders.filter(F.col("order_status")=="CLOSED").count()
print(result)


#Spark-sql

orders.createOrReplaceTempView("orders")
spark.sql("select count(*) from orders where order_status='CLOSED'").show()



51
+--------+
|count(*)|
+--------+
|      51|
+--------+



In [0]:
# Task 2.  Orders and revenue per month

# Add order_month (1-12) extracted from order_date
orders_with_month = orders.withColumn("order_month", F.month(F.to_date("order_date", "yyyy-MM-dd")))

orders_with_month.createOrReplaceTempView("orders_with_month")

result_df = (orders_with_month
    .groupBy("order_month")
    .agg(
        F.count("order_id").alias("num_orders"),
        F.round(F.sum(F.col("amount").cast("double")), 2).alias("total_revenue")
    )
    .orderBy("order_month")
)
result_df.show()



+-----------+----------+-------------+
|order_month|num_orders|total_revenue|
+-----------+----------+-------------+
|          7|       149|   2013549.66|
|          8|       151|   2008531.78|
+-----------+----------+-------------+



In [0]:
# Task 3.  Join orders with customers, total revenue per state (highest first)

# DataFrame API
joined = orders.join(customers, on="customer_id", how="inner")

result_state = (joined
    .groupBy("state")
    .agg(F.round(F.sum(F.col("amount").cast("double")), 2).alias("total_revenue"))
    .orderBy(F.col("total_revenue").desc())
)
display(result_state)

state,total_revenue
KA,994332.35
MH,772012.78
TN,771207.55
WB,409333.83
AP,370928.01
DL,359834.49
TG,248685.11


In [0]:
# Task 4.  Premium customers 

# How many customers have spent more than 100000 in total?
premium_count = (orders
    .groupBy("customer_id")
    .agg(F.sum(F.col("amount").cast("double")).alias("total_spent"))
    .filter(F.col("total_spent") > 100000)
    .count()
)
print(premium_count)

12


In [0]:

# Task 5.  Why no Dataset in PySpark?
#
# Dataset in Scala:
#   A Dataset is like a DataFrame but with strict type checking.
#   In Scala (which runs on the JVM), if you misspell a column name
#   like df.amout, the compiler catches it BEFORE the code runs.
#
# Why PySpark only has DataFrames:
#   Python is dynamically typed — there is no compiler that checks
#   column names ahead of time. So a typed Dataset would give no
#   extra safety in Python. PySpark keeps it simple with DataFrames
#   (just rows and columns, no fixed type per row).  
#
# When does df.amout get caught in PySpark?
#   NOT at compile time — only at RUNTIME.
#   When you call an action like .show() or .count(), Spark tries
#   to resolve the column name, fails, and throws an AnalysisException.

In [0]:
# Task 6.  String functions

result_str = customers.select(
    "customer_id",
    F.lower(F.concat_ws(".", F.col("first_name"), F.col("last_name"))).alias("email"),
    F.concat(F.substring(F.col("first_name"), 1, 1), F.substring(F.col("last_name"), 1, 1)).alias("initials"),
    F.upper(F.col("city")).alias("city_upper"),
    F.length(F.col("first_name")).alias("name_len")
)
display(result_str.limit(5))

customer_id,email,initials,city_upper,name_len
1,aarav.nair,AN,BENGALURU,5
2,vikram.reddy,VR,MUMBAI,6
3,varun.rao,VR,BENGALURU,5
4,rohan.patel,RP,HYDERABAD,5
5,neha.sharma,NS,MYSURU,4


In [0]:
# Task 7.  Numeric functions

result_num = orders.select(
    "order_id",
    "amount",
    F.round(F.col("amount").cast("double") * 1.18, 2).alias("amount_with_gst"),
    F.floor(F.col("amount").cast("double") / 1000).alias("amount_k")
)
display(result_num.limit(5))

order_id,amount,amount_with_gst,amount_k
1,9355.49,11039.48,9
2,6633.45,7827.47,6
3,6646.6,7842.99,6
4,2064.02,2435.54,2
5,22397.18,26428.67,22


In [0]:
# Task 9.  Conditional: amount bands

orders_banded = orders.withColumn(
    "amount_band",
    F.when(F.col("amount").isNull(), "UNKNOWN")
     .when(F.col("amount").cast("double") >= 20000, "HIGH")
     .when(F.col("amount").cast("double") >= 5000, "MEDIUM")
     .otherwise("LOW")
)

orders_banded.groupBy("amount_band").agg(
    F.count("order_id").alias("num_orders")
).orderBy(F.col("num_orders").desc()).show()

+-----------+----------+
|amount_band|num_orders|
+-----------+----------+
|     MEDIUM|       169|
|       HIGH|        69|
|        LOW|        52|
|    UNKNOWN|        10|
+-----------+----------+



In [0]:
# Task 11.  Aggregate functions per customer

customer_agg = (orders
    .groupBy("customer_id")
    .agg(
        F.count("order_id").alias("num_orders"),
        F.round(F.sum(F.col("amount").cast("double")), 2).alias("total_revenue"),
        F.round(F.avg(F.col("amount").cast("double")), 2).alias("avg_amount"),
        F.round(F.max(F.col("amount").cast("double")), 2).alias("largest_order"),
        F.countDistinct("order_status").alias("distinct_statuses"),
        F.collect_set("order_status").alias("status_set")
    )
    .orderBy(F.col("total_revenue").desc())
)
display(customer_agg.limit(5))

customer_id,num_orders,total_revenue,avg_amount,largest_order,distinct_statuses,status_set
24,10,161189.59,16118.96,33409.65,3,"List(CLOSED, ON_HOLD, COMPLETE)"
30,10,152688.93,15268.89,23835.35,6,"List(CANCELED, SUSPECTED_FRAUD, PROCESSING, PENDING, CLOSED, COMPLETE)"
11,9,149412.14,16601.35,39998.5,4,"List(CLOSED, PENDING, PENDING_PAYMENT, COMPLETE)"
5,9,147553.69,16394.85,25494.09,4,"List(PROCESSING, CLOSED, PENDING, COMPLETE)"
50,8,143036.02,17879.5,36008.47,4,"List(CLOSED, CANCELED, SUSPECTED_FRAUD, COMPLETE)"
